<a href="https://colab.research.google.com/github/fanat503/Induction-Heads-Tinystories/blob/main/induction_heads_tinystories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install tiktoken -q
import os
import time
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
B = 4
T = 1024
grad_accum_steps = 32
max_steps = 20000
warmup_steps = 200
max_lr = 6e-4
min_lr = 6e-5

project_dir = '***'
checkpoint_path = '***'

In [ ]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import time



class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        y = F.scaled_dot_product_attention(q,k,v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # output projection
        y = self.c_proj(y)
        return y


class MLP(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd, 4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
    self.c_proj.NANOGPT_SCALE_INIT = 1

  def forward(self,x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x


class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = MLP(config)

  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    x = x + self.mlp(self.ln_2(x))
    return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1

grad_accum_steps = 32

class GPT(nn.Module):

  def __init__(self, config):
    super().__init__()
    self.config = config

    self.transformer = nn.ModuleDict(dict(
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      wpe = nn.Embedding(config.block_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd),
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    self.transformer.wte.weight = self.lm_head.weight

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      std = 0.02
      if hasattr(module, 'NANOGPT_SCALE_INIT'):
        std *= (2 * self.config.n_layer) ** -0.5
      torch.nn.init.normal_(module.weight, mean = 0.0, std = std)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


  def forward(self,idx, targets=None):
    B,T = idx.size()
    assert T <= self.config.block_size
    pos = torch.arange(0, T, dtype = torch.long, device=idx.device)
    pos_emb = self.transformer.wpe(pos)
    tok_emb = self.transformer.wte(idx)
    x = tok_emb + pos_emb
    for block in self.transformer.h:
      x = block(x)

    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    loss = None
    if targets is not None:
      loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
    return logits, loss

In [ ]:
class DataLoaderLite:
    def __init__(self, B, T, split='train'):
        self.B = B
        self.T = T
        filename = os.path.join(project_dir, f'{split}.bin')

        self.tokens = np.memmap(filename, dtype=np.uint16, mode='r')
        print(f"[{split}] Downloaded {len(self.tokens):,} tokens")
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        buf = torch.tensor(buf.astype(np.int64), dtype=torch.long)

        x = (buf[:-1]).view(B, T)
        y = (buf[1:]).view(B, T)

        self.current_position += B * T
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0
        return x, y

def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_loader = DataLoaderLite(B=B, T=T, split='train')
val_loader = DataLoaderLite(B=B, T=T, split='val')
model = GPT(GPTConfig())
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda')

start_step = 0
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_step = checkpoint['step'] + 1
else:
    print("OOps")

for step in range(start_step, max_steps):
    t0 = time.time()

    if step > 0 and (step % 400 == 0 or step == max_steps - 1):
        model.eval()
        val_loss_accum = 0.0
        val_loss_steps = 20
        with torch.no_grad():
            for _ in range(val_loss_steps):
                x_val, y_val = val_loader.next_batch()
                x_val, y_val = x_val.to(device), y_val.to(device)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits, loss = model(x_val, y_val)
                val_loss_accum += loss.detach().item() / val_loss_steps
        print(f" VALIDATION step {step:4d} | val_loss: {val_loss_accum:.4f}")

        checkpoint = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'val_loss': val_loss_accum
        }
        torch.save(checkpoint, checkpoint_path)
        model.train()

    if step > 0 and step % 1000 == 0:
        checkpoint_path_step = os.path.join(
            project_dir, f'gpt2_mega_checkpoint{step}.pt'
        )
        torch.save(checkpoint, checkpoint_path_step)
        print(f"{checkpoint_path_step}")

    optimizer.zero_grad(set_to_none=True)
    loss_accum = 0.0

    for micro_step in range(grad_accum_steps):
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits, loss = model(x, y)
        loss = loss / grad_accum_steps
        loss_accum += loss.detach()
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    scaler.step(optimizer)
    scaler.update()

    if device == 'cuda':
      torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_processed = B * T * grad_accum_steps
    tokens_per_sec = tokens_processed / (t1 - t0)

    print(f"step {step:4d} | loss: {loss_accum.item():.4f} | lr: {lr:.4e} | norm: {norm:.2f} | dt: {dt:.2f}ms | tok/sec: {tokens_per_sec:.2f}")

In [ ]:
def load_checkpoint(path, model, device):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    return model


def make_repeated_sequence(tokenizer, length=50, device='cuda'):
    random_tokens = torch.randint(100, 5000, (length,))
    repeated = torch.cat([random_tokens, random_tokens])
    return repeated.unsqueeze(0).to(device)


def collect_attention_all_layers(model, input_ids):
    attention_maps = {}

    with torch.no_grad():
        x = model.transformer.wte(input_ids) + \
            model.transformer.wpe(
                torch.arange(input_ids.size(1), device=input_ids.device)
            )

        for layer_idx, block in enumerate(model.transformer.h):
            x_norm = block.ln_1(x)

            attn_out, attn_weights = block.attn.forward_with_attention(x_norm)

            attention_maps[layer_idx] = attn_weights.cpu()
            x = x + attn_out
            x = x + block.mlp(block.ln_2(x))

    return attention_maps

def compute_induction_score(attention_maps, seq_len_half):
    scores = {}
    for layer in attention_maps:
        for head in range(12):
            ind_pat = []
            for t in range(seq_len_half, seq_len_half * 2):
                source = t - seq_len_half + 1
                ind_pat.append(
                    attention_maps[layer][0, head, t, source]
                )
            ind_score = torch.tensor(ind_pat).mean()
            scores[(layer, head)] = ind_score
    return scores


def compute_previous_token_score(attention_maps):
    Score = {}
    for layer in attention_maps:
      for head in range(12):
        prev_tok_pat = []
        for t in range(1, 100):
            prev_tok_pat.append(attention_maps[layer][0, head, t, t-1])
        Score[(layer, head)] = torch.tensor(prev_tok_pat).mean()
    return Score

def analyze_checkpoint(path, seq_len_half, model, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    step = checkpoint['step']
    torch.manual_seed(42)

    random_tokens = torch.randint(100, 5000, (seq_len_half,))
    repeated = torch.cat([random_tokens, random_tokens])
    input_ids = repeated.unsqueeze(0).to(device)
    attention_maps = collect_attention_all_layers(model, input_ids)

    ih_scores = compute_induction_score(attention_maps, seq_len_half)
    pth_scores = compute_previous_token_score(attention_maps)
    def print_top_scores(scores, name, top_n=10):
      sorted_scores = sorted(
        scores.items(),
        key=lambda x: x[1].item(),
        reverse=True
        )
      for i, (key, score) in enumerate(sorted_scores[:top_n]):
        print(f"  Layer {key[0]:2d}, Head {key[1]:2d}: {score:.4f}")

    result = {
        'induction': ih_scores,
        'previous_token': pth_scores,
        'step': step
    }

    print(f"\n Step {step}")
    print_top_scores(ih_scores, "Induction Score")
    print_top_scores(pth_scores, "Previous Token Score")

    return result

def visualisation(scores, n_layer, n_head, title, step):
  grid = np.zeros((n_layer, n_layer))
  for layer in range(n_layer):
    for head in range(n_head):
      grid[layer, head] = scores[layer, head]
  plt.imshow(grid)
  plt.colorbar(cmap='hot')
  plt.xlabel("Head")
  plt.ylabel("Layer")
  plt.title(title)
  plt.xticks(range(n_head))
  plt.yticks(range(n_layer))
  plt.savefig(f'*{title}_{step}.png', dpi=300)
  plt.show()

@torch.no_grad()
def test_induction_score(model, seq_len_half=50):
    model.eval()

    pattern = list(range(1, seq_len_half + 1))
    full_seq = pattern + pattern
    x = torch.tensor([full_seq])

    attention_maps = {}

    def make_hook(layer_idx):
        def hook(module, input, output):
            pass
        return hook

    tok_emb = model.transformer.wte(x)
    pos = torch.arange(0, x.shape[1]).unsqueeze(0)
    pos_emb = model.transformer.wpe(pos)
    hidden = tok_emb + pos_emb

    for i, block in enumerate(model.transformer.h):
        attn_out, att = block.attn.forward_with_attention(block.ln_1(hidden))
        attention_maps[i] = att.detach().cpu()

        hidden = hidden + attn_out
        hidden = hidden + block.mlp(block.ln_2(hidden))

    scores = compute_induction_score(attention_maps, seq_len_half)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    for (layer, head), score in sorted_scores[:10]:
        print(f"  Layer {layer:2d}, Head {head:2d}: {score:.4f}")

    l6h4 = scores.get((6, 4), 0)
    print(f"\nL6H4: {l6h4:.4f}")

    if l6h4 > 0.1:
        print("*")
    elif l6h4 > 0.03:
        print("*")
    else:
        print("*")

    return scores

checkpoint = torch.load(
    '*',
    map_location='cpu'
)
model = GPT(GPTConfig())
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

In [ ]:
class SparseAutoEncoder(nn.Module):
    def __init__(self, d_model = 768, n_features = 8192):
        super().__init__()
        self.encoder = nn.Linear(d_model, n_features)
        self.decoder = nn.Linear(n_features, d_model, bias=False)
        self.pre_bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        x_centered = x - self.pre_bias
        hidden = F.relu(self.encoder(x_centered))
        reconstructed = self.decoder(hidden) + self.pre_bias
        return reconstructed, hidden

    def loss(self, x, reconstructed, hidden, l1_coeff=0.01):
        mse = F.mse_loss(reconstructed, x)
        l1 = hidden.abs().mean()
        total = mse + l1_coeff * l1
        return total, mse, l1

@torch.no_grad()
def collect_activations(model, dataloader, layer_idx=6, n_batches=100):
    all_activations = []
    activations = {}

    def hook_fn(module, input, output):
        activations['layer'] = output

    handle = model.transformer.h[layer_idx].register_forward_hook(hook_fn)
    model.eval()
    for i in range(n_batches):
        x, y = dataloader.next_batch()
        x = x.to(device)
        model = model.to(device)
        model(x)
        acts = activations['layer'].detach()
        acts = acts.view(-1, acts.shape[-1])
        all_activations.append(acts)

    handle.remove()
    return torch.cat(all_activations, dim=0)

def train_sae(activations, n_epochs=5, batch_size=4096, lr=3e-4, l1_coeff=0.01):
    sae = SparseAutoEncoder(d_model=768, n_features=8192).to(device)

    optimizer = torch.optim.AdamW(sae.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(activations)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(n_epochs):
      all_hidden = []
      total_loss = 0
      total_mse = 0
      n_batches = 0
      for (batch,) in loader:
          batch = batch.to(device)
          reconstructed, hidden = sae(batch)
          total, mse, l1 = sae.loss(batch, reconstructed, hidden, l1_coeff)

          optimizer.zero_grad()
          total.backward()
          optimizer.step()

          all_hidden.append(hidden.detach().cpu())
          total_loss += total.item()
          total_mse += mse.item()
          n_batches += 1

      all_hidden = torch.cat(all_hidden, dim=0)

      avg_active = ((all_hidden > 0).float().sum(dim=1)).mean()
      dead_features = (all_hidden.max(dim=0).values == 0).sum()

      print(f"Epoch {epoch+1}/{n_epochs} | "
            f"Loss: {total_loss/n_batches:.4f} | "
            f"MSE: {total_mse/n_batches:.4f} | "
            f"Active: {avg_active:.1f} | "
            f"Dead: {dead_features}/8192")

    return sae

@torch.no_grad()
def find_top_activating(sae, model, dataloader, feature_id,
                         n_batches=50, top_k=10, layer_idx=6):
    model.eval()
    results = []
    activations = {}

    def hook_fn(module, input, output):
        activations['layer'] = output

    handle = model.transformer.h[layer_idx].register_forward_hook(hook_fn)

    for i in range(n_batches):
        x, y = dataloader.next_batch()
        x = x.to(device)
        B, T = x.shape

        model(x)

        acts = activations['layer'].detach()
        acts = acts.view(-1, acts.shape[-1])

        reconstructed, hidden = sae(acts)
        score = hidden[:, feature_id]

        for pos in range(len(score)):
            if score[pos] > 0:
                b_idx = pos // T
                t_idx = pos % T

                context_start = max(0, t_idx - 5)
                context_end = min(T, t_idx + 3)
                context = x[b_idx, context_start:context_end].tolist()

                results.append({
                    'score': score[pos].item(),
                    'token_pos': t_idx,
                    'context': context
                })

    handle.remove()
    results.sort(key=lambda r: r['score'], reverse=True)
    return results[:top_k]

In [ ]:
def plot_feature_activations(model, sae, text, feature_ids, layer=6):
    tokens = tokenizer.encode(text)
    tokens_tensor = torch.tensor([tokens]).to(device)

    activations = {}
    def hook_fn(module, input, output):
        activations['resid'] = output[0].detach()

    handle = model.transformer.h[layer].register_forward_hook(hook_fn)

    with torch.no_grad():
        model(tokens_tensor)

    handle.remove()

    resid = activations['resid'].squeeze(0)
    with torch.no_grad():
        hidden = sae.encode(resid)

    token_labels = [tokenizer.decode([t]) for t in tokens]

    fig, axes = plt.subplots(len(feature_ids), 1, figsize=(max(12, len(tokens) * 0.4), 4 * len(feature_ids)))

    if len(feature_ids) == 1:
        axes = [axes]

    for ax, fid in zip(axes, feature_ids):
        values = hidden[:, fid].cpu().numpy()
        ax.bar(range(len(token_labels)), values, color='steelblue')
        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=8)
        ax.set_title(f'Feature #{fid} activations')
        ax.set_ylabel('Activation')

    plt.tight_layout()
    plt.savefig('sae_features.png', dpi=150, bbox_inches='tight')
    plt.show()

text = "Once upon a time there was a little girl. The little girl liked to play."
plot_feature_activations(model, sae, text, [8107, 635], layer=6)